# XLNet-base + Ridge Regression Comparison

Compare Official, Raw Whisper, and Trimmed Whisper transcripts using the same frozen XLNet-base encoder and the same Ridge Regression model. Ridge predicts the PHQ-8 score, which is then converted into five severity levels.

Official train and dev participants are used for training. Official test participants are used only for final evaluation.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
INPUT_PATH = PROJECT_ROOT / "input" / "transcript_model_input.csv"
RESULT_DIR = PROJECT_ROOT / "outputs" / "evaluation"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "xlnet/xlnet-base-cased"
MAX_LENGTH = 512
CHUNK_OVERLAP = 64
BATCH_SIZE = 8
RANDOM_STATE = 42

CLASS_IDS = [0, 1, 2, 3, 4]
CLASS_NAMES = ["None", "Mild", "Moderate", "Moderately severe", "Severe"]
TEXT_COLUMNS = {
    "Official": "official_text",
    "Raw Whisper": "raw_whisper_text",
    "Trimmed Whisper": "trimmed_whisper_text",
}
CACHE_NAMES = {
    "Official": "official",
    "Raw Whisper": "raw_whisper",
    "Trimmed Whisper": "trimmed_whisper",
}

## 2. Load the Prepared Input

In [ ]:
dataset = pd.read_csv(INPUT_PATH).sort_values("Participant_ID").reset_index(drop=True)
required_columns = {
    "Participant_ID",
    "Official_Split",
    "PHQ8_Severity",
    *TEXT_COLUMNS.values(),
}
missing_columns = required_columns - set(dataset.columns)
if missing_columns:
    raise ValueError(f"Missing input columns: {sorted(missing_columns)}")

assert dataset["Participant_ID"].is_unique
assert dataset[list(TEXT_COLUMNS.values())].notna().all().all()
assert dataset[list(TEXT_COLUMNS.values())].apply(lambda col: col.str.strip().ne("")).all().all()

train_mask = dataset["Official_Split"].isin(["train", "dev"]).to_numpy()
test_mask = dataset["Official_Split"].eq("test").to_numpy()
y_severity = dataset["PHQ8_Severity"].astype(int).to_numpy()
y_score = dataset["PHQ_8Total"].astype(float).to_numpy()

print(f"Total participants: {len(dataset)}")
print(f"Train + dev: {int(train_mask.sum())}")
print(f"Test: {int(test_mask.sum())}")
display(pd.crosstab(dataset["Official_Split"], dataset["PHQ8_Severity_Name"]))

## 3. Encode Long Transcripts with XLNet-base

Each transcript is split into overlapping token chunks. XLNet produces a contextual vector for every chunk, and chunk vectors are averaged into one participant vector. Embeddings are cached after the first run.

In [ ]:
def split_into_chunks(text: str, tokenizer) -> list[str]:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
    chunk_size = MAX_LENGTH - special_tokens
    step = chunk_size - CHUNK_OVERLAP
    if step <= 0:
        raise ValueError("CHUNK_OVERLAP must be smaller than chunk size")

    return [
        tokenizer.decode(token_ids[start:start + chunk_size], skip_special_tokens=True)
        for start in range(0, len(token_ids), step)
    ] or [""]


def fingerprint(documents: np.ndarray, participant_ids: np.ndarray) -> str:
    config = f"{MODEL_NAME}|max={MAX_LENGTH}|overlap={CHUNK_OVERLAP}"
    digest = hashlib.sha256(config.encode("utf-8"))
    for participant_id, document in zip(participant_ids, documents):
        digest.update(str(participant_id).encode("utf-8"))
        digest.update(document.encode("utf-8"))
    return digest.hexdigest()


def encode_corpus(
    documents: np.ndarray,
    participant_ids: np.ndarray,
    cache_name: str,
) -> np.ndarray:
    cache_path = RESULT_DIR / f"xlnet_base_{cache_name}_embeddings.npz"
    current_fingerprint = fingerprint(documents, participant_ids)

    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        if cached["fingerprint"].item() == current_fingerprint:
            print(f"Loaded cache: {cache_path.name}")
            return cached["embeddings"]

    document_embeddings = []
    for document in tqdm(documents, desc=f"Encoding {cache_name}"):
        chunks = split_into_chunks(document, tokenizer)
        chunk_batches = []

        for start in range(0, len(chunks), BATCH_SIZE):
            inputs = tokenizer(
                chunks[start:start + BATCH_SIZE],
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(device)

            with torch.inference_mode():
                token_embeddings = encoder(**inputs).last_hidden_state

            attention_mask = inputs["attention_mask"].unsqueeze(-1)
            chunk_embeddings = (token_embeddings * attention_mask).sum(dim=1) / (
                attention_mask.sum(dim=1).clamp(min=1)
            )
            chunk_embeddings = torch.nn.functional.normalize(
                chunk_embeddings, p=2, dim=1
            )
            chunk_batches.append(chunk_embeddings.cpu().numpy())

        pooled = np.vstack(chunk_batches).mean(axis=0)
        pooled /= max(np.linalg.norm(pooled), 1e-12)
        document_embeddings.append(pooled)

    embeddings = np.vstack(document_embeddings)
    np.savez_compressed(
        cache_path, embeddings=embeddings, fingerprint=np.array(current_fingerprint)
    )
    return embeddings


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
encoder.eval()

participant_ids = dataset["Participant_ID"].to_numpy()
embeddings = {
    source: encode_corpus(
        dataset[text_column].astype(str).to_numpy(),
        participant_ids,
        CACHE_NAMES[source],
    )
    for source, text_column in TEXT_COLUMNS.items()
}

print(f"Device: {device}")
print(f"Embedding shape: {next(iter(embeddings.values())).shape}")

## 4. Train the Same Ridge Regressor for All Three Inputs

XLNet remains frozen. Ridge Regression predicts the continuous PHQ-8 score from 0 to 24. Predicted scores are clipped to this valid range and converted to the five severity levels.

In [ ]:
def make_regressor() -> Pipeline:
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "regressor",
                Ridge(alpha=10.0),
            ),
        ]
    )


metric_rows = []
predictions = dataset.loc[
    test_mask, ["Participant_ID", "PHQ_8Total", "PHQ8_Severity"]
].copy()
train_scores = y_score[train_mask]
test_scores = y_score[test_mask]
test_severity = y_severity[test_mask]

for source, features in embeddings.items():
    regressor = make_regressor()
    regressor.fit(features[train_mask], train_scores)
    predicted_score = regressor.predict(features[test_mask]).clip(0, 24)
    predicted_severity = np.digitize(
        predicted_score, bins=[5, 10, 15, 20], right=False
    )
    predictions[f"{source}_predicted_score"] = predicted_score
    predictions[f"{source}_prediction"] = predicted_severity

    metric_rows.append(
        {
            "input": source,
            "mae": mean_absolute_error(test_scores, predicted_score),
            "rmse": mean_squared_error(test_scores, predicted_score) ** 0.5,
            "r2": r2_score(test_scores, predicted_score),
            "accuracy": accuracy_score(test_severity, predicted_severity),
            "balanced_accuracy": balanced_accuracy_score(
                test_severity, predicted_severity
            ),
            "precision_macro": precision_score(
                test_severity, predicted_severity, average="macro", zero_division=0
            ),
            "recall_macro": recall_score(
                test_severity, predicted_severity, average="macro", zero_division=0
            ),
            "f1_macro": f1_score(
                test_severity, predicted_severity, average="macro", zero_division=0
            ),
        }
    )

results = pd.DataFrame(metric_rows).set_index("input")
display(results.round(3))

## 5. Basic Result Plots

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
results[["mae", "rmse"]].plot(kind="bar", rot=0, ax=axes[0])
axes[0].set_title("PHQ-8 score error (lower is better)")
axes[0].set_ylabel("Points")
axes[0].grid(axis="y", alpha=0.25)
results[["accuracy", "balanced_accuracy", "f1_macro"]].plot(
    kind="bar", ylim=(0, 1), rot=0, ax=axes[1]
)
axes[1].set_title("Five-level severity classification")
axes[1].set_ylabel("Score")
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, source in zip(axes, TEXT_COLUMNS):
    ConfusionMatrixDisplay.from_predictions(
        test_severity,
        predictions[f"{source}_prediction"],
        labels=CLASS_IDS,
        display_labels=CLASS_NAMES,
        cmap="Blues",
        colorbar=False,
        ax=axis,
    )
    axis.set_title(source)
    axis.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. Save Results

In [ ]:
results.to_csv(RESULT_DIR / "xlnet_ridge_input_comparison.csv")
predictions.to_csv(RESULT_DIR / "xlnet_ridge_test_predictions.csv", index=False)
print(f"Saved results to: {RESULT_DIR}")